<a href="https://colab.research.google.com/github/tkme1981-hero/RFM-Project/blob/main/20261001RFM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import io
import chardet
from IPython.display import display
import pandas as pd
import requests


def get_raw_bytes(source: str) -> bytes:
    """判斷是網址或本機路徑，並回傳 bytes"""
    source = source.strip().strip("'\"")

    if source.startswith("http://") or source.startswith("https://"):
        # 若為 GitHub 網頁連結，自動轉成 raw 資料網址
        if "github.com" in source and "/blob/" in source:
            source = source.replace("github.com", "raw.githubusercontent.com").replace(
                "/blob/", "/"
            )

        headers = {"User-Agent": "Mozilla/5.0"}
        response = requests.get(source, headers=headers, timeout=15)
        response.raise_for_status()
        return response.content
    else:
        with open(source, "rb") as f:
            return f.read()


def load_csv_with_fallback(data_bytes: bytes) -> pd.DataFrame:
    """依序嘗試常見中文編碼，失敗則使用 chardet 自動偵測"""
    candidate_encodings = ["utf-8-sig", "utf-8", "cp950", "big5"]

    for enc in candidate_encodings:
        try:
            df = pd.read_csv(io.BytesIO(data_bytes), encoding=enc)
            print(f"成功使用編碼讀取：【{enc}】")
            return df
        except (UnicodeDecodeError, UnicodeError):
            continue

    # 自動偵測編碼
    detected = chardet.detect(data_bytes[:20000])
    detected_enc = detected.get("encoding")
    print(f"一般編碼嘗試失敗，chardet 偵測為：【{detected_enc}】")

    if detected_enc:
        try:
            return pd.read_csv(io.BytesIO(data_bytes), encoding=detected_enc)
        except Exception:
            pass

    # 最終容錯
    print("注意：使用備用容錯模式 (errors='replace') 讀取")
    return pd.read_csv(
        io.BytesIO(data_bytes), encoding="utf-8", encoding_errors="replace"
    )


# ---------------- 執行區 ----------------
# 提示使用者輸入，輸入完直接按「Enter」即可執行
source_input = input("請輸入 CSV 網址或本機路徑 (按 Enter 確認): ").strip()

if not source_input:
    print("您沒有輸入任何路徑或網址！")
else:
    try:
        print(f"正在讀取資料：{source_input} ...")
        raw_data = get_raw_bytes(source_input)
        df = load_csv_with_fallback(raw_data)

        # 處理日期欄位：檢查是否有 Date 欄位，若無則預設第 1 欄為日期
        date_col = (
            "Date"
            if "Date" in df.columns
            else (df.columns[0] if len(df.columns) > 0 else None)
        )

        if date_col:
            df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
            print(f"已將欄位【{date_col}】轉換為日期格式 (Datetime)")

        print("\n讀取成功！前五列資料如下：")
        display(df.head(5))

    except FileNotFoundError:
        print("【錯誤】找不到檔案，請確認本機路徑是否正確（例如 D:/data.csv）。")
    except requests.exceptions.RequestException as e:
        print(f"【網路錯誤】下載失敗，請確認網址是否可公開存取：{e}")
    except Exception as e:
        print(f"【發生未預期錯誤】：{e}")

In [ ]:
# 1. 確保日期欄位被正確解析（請將 'Date' 換成你 CSV 裡的實際欄位名）
df['Date'] = pd.to_datetime(df['Date'])

# 2. 以資料庫中的最大日期作為基準點，用來計算 Recency
max_date = df['Date'].max()

# 3. 核心轉換：利用 groupby 進行客戶別聚合計算
# （請根據你 CSV 實際的欄位名稱，修改 'CustomerKey'、'InvoNo'、'Amount'）
rfm_df = df.groupby('CustomerKey').agg({
    'Date': lambda x: (max_date - x.max()).days,  # Recency：最後一次交易離基準點幾天
    'InvoNo': 'nunique',                        # Frequency：獨特的發票編號總數
    'Amount': 'sum'                                # Monetary：Amount總加
}).reset_index()

# 4. 修改欄位名稱，讓表格一目了然
rfm_df.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

# 5. 直接存在本機（存成無痛的 UTF-8 編碼，不帶索引欄位）
rfm_df.to_csv("RFM_Result.csv", index=False, encoding="utf-8-sig")

# 6. 秀出成果
rfm_df.head()

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# 選擇要正規化的欄位
features = ['Recency', 'Frequency', 'Monetary']

# 初始化 Min-Max Scaler
scaler = MinMaxScaler()

# 對選定的欄位進行正規化
rfm_n_scaled = scaler.fit_transform(rfm_df[features])

# 將正規化後的數據轉換回 DataFrame，並保留 CustomerID
rfm_n = pd.DataFrame(rfm_n_scaled, columns=features)
rfm_n['CustomerID'] = rfm_df['CustomerID']

# 重新排列欄位順序，將 CustomerID 放回第一位
rfm_n = rfm_n[['CustomerID', 'Recency', 'Frequency', 'Monetary']]

print("RFM 正規化後的資料 (rfm_n) 前五列：")
display(rfm_n.head())

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# 準備用於分群的數據 (不包含 CustomerID)
X = rfm_n[['Recency', 'Frequency', 'Monetary']]

# 計算不同 k 值下的 WCSS (Within-Cluster Sum of Squares)
wcss = []
for i in range(1, 11): # 嘗試 1 到 10 個分群
    kmeans = KMeans(n_clusters=i, init='k-means++', random_state=42, n_init='auto')
    kmeans.fit(X)
    wcss.append(kmeans.inertia_)

# 繪製 Elbow chart
plt.figure(figsize=(10, 6))
plt.plot(range(1, 11), wcss, marker='o', linestyle='--')
plt.title('Elbow Method for Optimal k')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('WCSS')
plt.grid(True)
plt.xticks(range(1, 11))
plt.show()

print("請觀察 Elbow chart 的 '手肘' 點，該點通常代表最佳的分群數 (k 值)。")

In [ ]:
from sklearn.cluster import KMeans

# 準備用於分群的數據 (不包含 CustomerID)
X_kmeans = rfm_n[['Recency', 'Frequency', 'Monetary']]

# 設定分群數為 4
k = 4

# 執行 K-Means 分群
kmeans_model = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init='auto')
rfm_n['Cluster'] = kmeans_model.fit_predict(X_kmeans)

print(f"以 k={k} 執行 K-Means 分群後的 rfm_n (前五列)：")
display(rfm_n.head())

In [ ]:
# 將分群標籤 'Cluster' 從 rfm_n 合併回 rfm_df
rfm_df = pd.merge(rfm_df, rfm_n[['CustomerID', 'Cluster']], on='CustomerID', how='left')

print("rfm_df 合併分群標籤後的資料 (前五列)：")
display(rfm_df.head())

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 繪製 Recency, Frequency, Monetary 的成對散布圖，並依 Cluster 著色
sns.pairplot(rfm_df, vars=['Recency', 'Frequency', 'Monetary'], hue='Cluster', palette='tab10')
plt.suptitle('Pair Plot of RFM by Cluster', y=1.02) # y 調整標題位置，避免與圖重疊
plt.show()

In [ ]:
# 定義 Cluster 到 Group 的映射關係
cluster_to_group = {
    0: 'Missing',
    1: 'H',
    2: 'M',
    3: 'L'
}

# 創建新的 'Group' 欄位
rfm_df['Group'] = rfm_df['Cluster'].map(cluster_to_group)

print("rfm_df 新增 'Group' 欄位後的資料 (前五列)：")
display(rfm_df.head())

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 繪製 Recency, Frequency, Monetary 的成對散布圖，並依 Group 著色
g = sns.pairplot(rfm_df, vars=['Recency', 'Frequency', 'Monetary'], hue='Group', palette='tab10')
plt.suptitle('Pair Plot of RFM by Group', y=1.02) # y 調整標題位置，避免與圖重疊

# 儲存圖表為透明背景的 PNG 檔案
plt.savefig('RFM.png', transparent=True, bbox_inches='tight')

plt.show()

print("RFM 成對散布圖已繪製並儲存為 RFM.png")

In [ ]:
# 將 rfm_df 匯出為 CSV 檔案
rfm_df.to_csv('RFM_Grouped.csv', index=False, encoding='utf-8-sig')

print("rfm_df 已成功匯出為 RFM_Grouped.csv")